Step 1: Install packages

In [0]:
%pip install -U \
langchain \
langchain-community \
langchain-openai \
langchain-text-splitters \
langchain-huggingface \
chromadb \
sentence-transformers \
pypdf \
openai

In [0]:
#Restart Python
dbutils.library.restartPython()

Step 2: Load PDF

In [0]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/Volumes/workspace/default/tmp/Reliance Q125 earnings transcript.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()

print(f"Total Pages: {len(docs)}")

Step 3: Split PDF into chunks

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

print(f"Total Chunks: {len(documents)}")

Step 4: Create Embeddings

In [0]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Step 5: Create Chroma Vector Database

In [0]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="/tmp/chroma_db"
)

print("Vector Store Created Successfully!")

Step 6: Create Retriever

In [0]:
retriever = db.as_retriever(
    search_kwargs={"k":5}
)

Step 7: Test Retrieval

In [0]:
query = "Summarize the key highlights of Reliance Q1 FY26 earnings"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print("="*80)
    print(f"Chunk {i+1}")
    print(doc.page_content[:500])

Step 8: Connect Databricks Llama 4 Maverick

In [0]:
from openai import OpenAI

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url = "https://dbc-9f49578e-aff4.cloud.databricks.com/serving-endpoints"
)

Step 9: Create LLM Function

In [0]:
def get_llm_answer(query, reference_data):

    response = client.chat.completions.create(
        model="databricks-llama-4-maverick",

        messages=[
            {
                "role":"system",
                "content":"""
You are an expert financial AI assistant.

Answer ONLY using the supplied reference.

If the information is unavailable, say:
'I could not find that information in the provided document.'

Provide:

• Executive Summary

• Key Positives

• Key Risks

• Investment Outlook

• Final Conclusion

Use bullet points and suitable emojis.
"""
            },

            {
                "role":"user",
                "content":f"""
Question:

{query}

Reference:

{reference_data}
"""
            }

        ],

        temperature=0.2,
        max_tokens=1000
    )

    return response.choices[0].message.content

Step 10: Ask Questions

In [0]:
query = """
Can you summarize the key highlights of the earnings call?

Based ONLY on the transcript:

1. Executive Summary

2. Positives

3. Risks

4. Revenue Growth

5. EBITDA

6. Jio Business

7. Retail Business

8. New Energy

9. Whether management outlook is Positive, Neutral or Negative.

Do NOT provide investment advice.
"""

retrieved_docs = retriever.invoke(query)

reference = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

llm_result = get_llm_answer(
    query=query,
    reference_data=reference
)

print(llm_result)

Step 11: Simple Chat

In [0]:
query = input("Ask a question: ")

retrieved_docs = retriever.invoke(query)

reference = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

answer = get_llm_answer(
    query=query,
    reference_data=reference
)

print(answer)

Final Architecture

PDF
 │
 ▼
PyPDFLoader
 │
 ▼
RecursiveCharacterTextSplitter
 │
 ▼
Sentence Transformer
(all-MiniLM-L6-v2)
 │
 ▼
Chroma Vector DB
 │
 ▼
Retriever
 │
 ▼
Relevant Chunks
 │
 ▼
Databricks Llama-4-Maverick
 │
 ▼
Final Answer